In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import multiprocessing

print(f"CPU cores: {multiprocessing.cpu_count()}")
print(f"RAM: {os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / (1024**3):.1f} GB")

CPU cores: 8
RAM: 16.0 GB


In [3]:
import numpy as np
import h5py
import matplotlib.pyplot as plt
from numba import jit
import scipy
import pandas as pd

In [4]:
planets = pd.read_csv("../input/planets_triple.csv")

In [5]:
triples = pd.read_csv('../input/triple_systems_selected.csv')

In [6]:
from integrator import run_system
from integrator.experiments import generate_habitability_experiments

In [7]:
configs = generate_habitability_experiments(
    triples,
    planets,
    tf=1e4,
    output_dir='output',
)

print(f"Generated {len(configs)} experiment configs")
print(configs[0]['experiment_name'])
print(configs[0]['internal_to_physical'])

Generated 349 experiment configs
Gliese_667_observed_ev0.0_iv000
{'A': 'A', 'B': 'B', 'C': 'C'}


In [8]:
preview_df = pd.DataFrame([
    {
        'system_name': cfg['system_name'],
        'planet_scenario': cfg['planet_scenario'],
        'host_star': cfg['host_star'],
        'system_type': cfg['system_type'],
        'outer_e': cfg['outer_e'],
        'outer_i': cfg['outer_i'],
        'source_outer_e': cfg['source_outer_e'],
        'source_outer_i': cfg['source_outer_i'],
        'source_inner_e': cfg['source_inner_e'],
        'source_inner_i': cfg['source_inner_i'],
        'n_planets': len(cfg['planets']),
        'experiment_name': cfg['experiment_name'],
        'output_file': cfg['output_file'],
    }
    for cfg in configs
])

summary_df = (
    preview_df.groupby(['system_name', 'planet_scenario'], as_index=False)
    .agg(
        n_runs=('experiment_name', 'size'),
        outer_e_values=('outer_e', lambda s: sorted(pd.unique(s))),
        outer_i_values=('outer_i', lambda s: sorted(pd.unique(s))),
        n_planets=('n_planets', 'first'),
    )
    .sort_values(['system_name', 'planet_scenario'])
)

print(f"Total configs: {len(preview_df)}")
display(summary_df)
display(preview_df.head(20))

Total configs: 349


,system_name,planet_scenario,n_runs,outer_e_values,outer_i_values,n_planets
0,94 Ceti,hz_inner,1,[0.26],[104.0],1
1,94 Ceti,hz_mid,1,[0.26],[104.0],1
2,94 Ceti,hz_outer,1,[0.26],[104.0],1
3,GJ 229,hz_inner,1,[0.736],[47.7],1
4,GJ 229,hz_mid,1,[0.736],[47.7],1
5,GJ 229,hz_outer,1,[0.736],[47.7],1
6,GJ 900 A,hz_inner,20,"[0.0, 0.2, 0.4, 0.6, 0.8]","[0.0, 30.0, 60.0, 90.0]",1
7,GJ 900 A,hz_mid,20,"[0.0, 0.2, 0.4, 0.6, 0.8]","[0.0, 30.0, 60.0, 90.0]",1
8,GJ 900 A,hz_outer,20,"[0.0, 0.2, 0.4, 0.6, 0.8]","[0.0, 30.0, 60.0, 90.0]",1
9,Gliese 667,observed,20,"[0.0, 0.2, 0.4, 0.6, 0.8]","[0.0, 30.0, 60.0, 90.0]",5


,system_name,planet_scenario,host_star,system_type,outer_e,outer_i,source_outer_e,source_outer_i,source_inner_e,source_inner_i,n_planets,experiment_name,output_file
0,Gliese 667,observed,C,S(C),0.0,0.0,NaN,NaN,0.57,127.6,5,Gliese_667_observed_ev0.0_iv000,output/Gliese_667_observed_ev0.0_iv000.hdf5
1,Gliese 667,observed,C,S(C),0.0,30.0,NaN,NaN,0.57,127.6,5,Gliese_667_observed_ev0.0_iv030,output/Gliese_667_observed_ev0.0_iv030.hdf5
2,Gliese 667,observed,C,S(C),0.0,60.0,NaN,NaN,0.57,127.6,5,Gliese_667_observed_ev0.0_iv060,output/Gliese_667_observed_ev0.0_iv060.hdf5
3,Gliese 667,observed,C,S(C),0.0,90.0,NaN,NaN,0.57,127.6,5,Gliese_667_observed_ev0.0_iv090,output/Gliese_667_observed_ev0.0_iv090.hdf5
4,Gliese 667,observed,C,S(C),0.2,0.0,NaN,NaN,0.57,127.6,5,Gliese_667_observed_ev0.2_iv000,output/Gliese_667_observed_ev0.2_iv000.hdf5
5,Gliese 667,observed,C,S(C),0.2,30.0,NaN,NaN,0.57,127.6,5,Gliese_667_observed_ev0.2_iv030,output/Gliese_667_observed_ev0.2_iv030.hdf5
6,Gliese 667,observed,C,S(C),0.2,60.0,NaN,NaN,0.57,127.6,5,Gliese_667_observed_ev0.2_iv060,output/Gliese_667_observed_ev0.2_iv060.hdf5
7,Gliese 667,observed,C,S(C),0.2,90.0,NaN,NaN,0.57,127.6,5,Gliese_667_observed_ev0.2_iv090,output/Gliese_667_observed_ev0.2_iv090.hdf5
8,Gliese 667,observed,C,S(C),0.4,0.0,NaN,NaN,0.57,127.6,5,Gliese_667_observed_ev0.4_iv000,output/Gliese_667_observed_ev0.4_iv000.hdf5
9,Gliese 667,observed,C,S(C),0.4,30.0,NaN,NaN,0.57,127.6,5,Gliese_667_observed_ev0.4_iv030,output/Gliese_667_observed_ev0.4_iv030.hdf5
